<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/OCR_bigc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Method 2: Tesseract OCR

In [1]:
!pip install pytesseract pillow requests easyocr playwright nest_asyncio
!playwright install chromium
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 20.7 MB/s eta 0:00:00
(node:10951) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 344.7s167.3 MiB [] 0% 43.9s167.3 MiB [] 0% 18.0s167.3 MiB [] 0% 14.8s167.3 MiB [] 0% 12.5s167.3 MiB [] 1% 6.5s167.3 MiB [] 2% 4.8s167.3 MiB [] 2% 3.9s167.3 MiB [] 3% 3.5s167.3 MiB [] 3% 3.7s167.3 MiB [] 4% 3.7s167.3 MiB [] 5% 3.3s167.3 MiB [] 6% 2.9s167.3 MiB [] 7% 2.7s167.3 MiB [] 8

In [4]:
import io
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
import requests
import easyocr
from PIL import Image

# Apply the fix for running inside a notebook
nest_asyncio.apply()

# --- 1. YOUR LIST OF PRODUCT URLS ---
PRODUCT_URLS = [
    "https://www.bigc.co.th/en/product/masita-korean-style-roasted-seaweed-original-flavour-size-12-g.18534",
    "https://www.bigc.co.th/en/product/shokubutsu-shower-cream-light-green-color-400-ml.3137",
    "https://www.bigc.co.th/en/product/grainey-mini-multi-grain-bar-honey-and-almond-flavour-20g.90693"
]

async def get_badges_with_playwright(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-dev-shm-usage'])
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        print(f"\n🔍 Navigating to: {url}")
        try:
            # We use 'domcontentloaded' to get in fast, then wait for the element
            await page.goto(url, wait_until="domcontentloaded", timeout=45000)

            # Wait specifically for the badge to exist in the code
            print("   Waiting for badge elements...")
            await page.wait_for_selector('img[alt="product badge"]', state="attached", timeout=10000)

            # Brief pause for lazy-loading to resolve
            await asyncio.sleep(2)

            badge_elements = await page.query_selector_all('img[alt="product badge"]')

            urls = []
            for img in badge_elements:
                src = await img.get_attribute('src')
                if src:
                    urls.append(src)

            # Return unique URLs only
            return list(set(urls))

        except Exception as e:
            print(f"   ⚠️ Could not find badges on this page: {e}")
            return []
        finally:
            await browser.close()

def run_ocr_on_badges(url_list):
    if not url_list:
        print("   ❌ No badge URLs found to process.")
        return

    print(f"   ✅ Found {len(url_list)} unique badge(s). Starting OCR...")

    # Initialize reader for both English and Thai to ensure we don't miss anything
    reader = easyocr.Reader(['en', 'th'])
    img_headers = {"User-Agent": "Mozilla/5.0", "Referer": "https://www.bigc.co.th/"}

    for i, badge_url in enumerate(url_list):
        print(f"   --- Processing Badge {i+1} ---")
        try:
            response = requests.get(badge_url, headers=img_headers, timeout=10)
            if response.status_code == 200:
                img = Image.open(io.BytesIO(response.content))

                # Upscale 4x for better OCR readability
                img = img.resize((img.width * 4, img.height * 4), resample=Image.LANCZOS)

                filename = f"temp_badge.png"
                img.save(filename)

                results = reader.readtext(filename)
                final_text = " ".join([res[1] for res in results]).upper()
                print(f"   OCR Result: {final_text}")

                # Improved keyword check: includes Thai "Buy" (ซื้อ) and "Free/Get" (แถม)
                promo_keywords = ["BUY", "GET", "FREE", "1", "2", "ซื้อ", "แถม"]
                if any(k in final_text for k in promo_keywords):
                    print("   🎯 SUCCESS: Promotion text detected!")
                else:
                    print("   ℹ️ No standard promotion text found.")
            else:
                print(f"   ❌ Image Download Failed (Status: {response.status_code})")
        except Exception as e:
            print(f"   ❌ OCR Error: {e}")

async def main():
    # --- LOOP THROUGH THE LIST ---
    for target_url in PRODUCT_URLS:
        badge_urls = await get_badges_with_playwright(target_url)
        run_ocr_on_badges(badge_urls)

    print("\n🏁 All URLs processed.")

# Execute
await main()


🔍 Navigating to: https://www.bigc.co.th/en/product/masita-korean-style-roasted-seaweed-original-flavour-size-12-g.18534
   Waiting for badge elements...


   ✅ Found 1 unique badge(s). Starting OCR...


Progress: |██████████████████████████████████████████████████| 100.0% Complete   --- Processing Badge 1 ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   OCR Result: ซื้อ2  1 แถม
   🎯 SUCCESS: Promotion text detected!

🔍 Navigating to: https://www.bigc.co.th/en/product/shokubutsu-shower-cream-light-green-color-400-ml.3137
   Waiting for badge elements...


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   OCR Result: ซื้อ1 แถม
   🎯 SUCCESS: Promotion text detected!

🔍 Navigating to: https://www.bigc.co.th/en/product/grainey-mini-multi-grain-bar-honey-and-almond-flavour-20g.90693
   Waiting for badge elements...


   ✅ Found 1 unique badge(s). Starting OCR...
   --- Processing Badge 1 ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   OCR Result: ซื้อ2  1 แถม
   🎯 SUCCESS: Promotion text detected!

🏁 All URLs processed.
